In [2]:
# import libraries
import pandas as pd
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# load the data
all_posts_df = pd.read_csv("../01_data/clean_data/all_posts_cleaned.csv")

In [5]:
# load cardiff nlp roberta model
model = AutoModelForSequenceClassification.from_pretrained('cardiffnlp/xlm-roberta-base-tweet-sentiment-de')
tokenizer= AutoTokenizer.from_pretrained('cardiffnlp/xlm-roberta-base-tweet-sentiment-de')

In [8]:
# create function to label single post
def label_post(post, model, tokenizer):

    # get token ids and run them through the model
    input_ids = tokenizer(post, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**input_ids)

    # turn logits to probabilities and extract probs for positive and negative class
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    positive_prob = probs[0, 2].item()
    negative_prob = probs[0, 0].item()

    # take the log of the ratio (log-odds) with a smoothing factor
    alpha = 1e-6
    log_ratio = np.log((positive_prob + alpha) / (negative_prob + alpha))
    return log_ratio

# apply the function to the full df
all_posts_df["sentiment"] = all_posts_df["text"].apply(lambda x: label_post(x, model, tokenizer))

# show first few rows
all_posts_df.head()

KeyboardInterrupt: 

In [ ]:
# group by each MP by one week intervals and take the mean sentiment score

In [ ]:
# export the labelled data
all_posts_df.to_csv("../01_data/clean_labelled_data/all_posts_cleaned_labelled.csv")

# export the grouped and labelled data
all_posts_df.to_csv("../01_data/clean_labelled_data/all_posts_cleaned_labelled.csv")